# Configuration

In [1]:
import os 

if True ^ os.getcwd().endswith('hte-and-targeting'):
    os.chdir('..')

In [2]:
import pandas as pd 
import numpy as np

In [3]:
from scipy.stats import norm, invgamma
# estimators
# from econml.grf import CausalForest
from statsmodels.regression.linear_model import OLS, WLS

from scipy.integrate import quad 
from scipy.stats import ks_2samp

In [4]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
import plotly.express as px
import plotly.graph_objects as go

plt.rcParams['text.usetex'] = False

In [5]:
from core.variables import * 
from core.pricing_estimators import *
from core.dgp import *
from core.experiments import *
from core.visualization import *

# Experiments

In [21]:
class PersonalizedPricingDGP(object):
    def __init__(self, cov_dim: int = 1):
        self.cov_dim = cov_dim
        self.util_const_map = np.random.uniform(0, 1, size=(cov_dim, 1))  # (cov_dim, 1)
        self.util_price_map = np.random.uniform(-0.1, 0, size=(cov_dim, 1))  # (cov_dim, 1)

    def generate_training_data(
        self, sample_size: int, price_lb: float, price_ub: float, price_diff: float, seed=None
    ) -> tuple:
        """
        Generate training data

        Params:
        -------
        sample_size: int, number of samples to generate
        price_lb: float, lower bound of the price
        price_ub: float, upper bound of the price
        price_diff: float, price difference
        seed: int, random seed

        Returns:
        -------
        tuple: 
            - X: np.ndarray, covariate
            - T: np.ndarray, treatment assignment
            - Y: np.ndarray, outcome
        """
        # generate data
        # individual characteristics
        X_arr = self.__generate_individual_characteristics(sample_size)  # (sample_size, cov_dim)
        
        # price (treaments)
        price_arr = np.random.choice(np.arange(price_lb, price_ub, price_diff), sample_size).reshape(-1, 1)  # (sample_size, 1)

        # error
        err_arr = np.random.gumbel(loc=0, scale=1, size=(sample_size, 1))  # (sample_size, 1)

        # utility: (sample_size, 1)
        util_arr = X_arr @ self.util_const_map + X_arr @ self.util_price_map * price_arr + err_arr

        # demand: (sample_size, 1), buy if utility > 0, else don't buy.
        demand_arr = (util_arr > 0).astype(int)

        return X_arr, price_arr, demand_arr
    
    def generate_testing_data(self, sample_size: int) -> np.ndarray:        
        return self.__generate_individual_characteristics(sample_size)  # (sample_size, )

    def __generate_individual_characteristics(self, sample_size: int) -> np.ndarray:
        return np.random.normal(loc=1, scale=0.1, size=(sample_size, self.cov_dim))

In [22]:
# parameters
train_size = 500
test_size = 10
cov_dim = 5

price_lb, price_ub, price_diff = 1, 10, 1

# initialize the data generating process
dgp = PersonalizedPricingDGP(cov_dim=cov_dim)

cov_arr, price_arr, outcome_arr = dgp.generate_training_data(
    sample_size=train_size, price_lb=price_lb, price_ub=price_ub, price_diff=price_diff
)
test_cov_arr = dgp.generate_testing_data(sample_size=test_size)

# train estimators 
# plugin estimator
plugin_estmr = PricingPlugIn(cov_dim=cov_dim).fit(covariates=cov_arr, prices=price_arr, outcomes=outcome_arr)

Optimization terminated successfully.
         Current function value: 0.165072
         Iterations 10


In [23]:
def evaluate_pricing_policy(
    covariates: np.ndarray, price: float, dgp: PersonalizedPricingDGP, delta: float = 0.99
) -> float:
    """
    Evaluate the pricing policy

    Params:
    -------
    covariates: np.ndarray, individual characteristics
    price: float, price
    dgp: PersonalizedPricingDGP, data generating process
    delta: float, discount factor

    Returns:
    -------
    float: expected profit
    """

    logit = np.exp(covariates @ dgp.util_const_map + (price * covariates) @ dgp.util_price_map)
    purchase_prob = logit / (1 + logit)

    return np.mean(price * purchase_prob / (1 - delta * purchase_prob))

In [24]:
# evalute test set
# plugin estimator and its performance
plugin_targ_est = plugin_estmr.estimate_targeting_value(covariates=test_cov_arr)
true_plugin_targ_val = evaluate_pricing_policy(covariates=test_cov_arr, price=plugin_estmr.get_targeting_policy(test_cov_arr, delta=0.99), dgp=dgp)

print(f"Plugin targeting estimate: {plugin_targ_est:.4f}")

print(f"\nActual value of plugin targeting: {true_plugin_targ_val:.4f}")
print(f"Winner's Curse of plugin estimate: {plugin_targ_est - true_plugin_targ_val:.4f}")

Plugin targeting estimate: 289.7828

Actual value of plugin targeting: 25.5002
Winner's Curse of plugin estimate: 264.2826


In [ ]:
class PricingValueCorrection(PricingPlugIn):
    def __init__(self, cov_dim: int):
        # attributes
        self.cov_dim = cov_dim

        # place holders
        self.n_bootstraps = None  # number of bootstrap samples
        self.boot_util_const_map = None  # (n_bootstraps, cov_dim)
        self.boot_util_price_map = None  # (n_bootstraps, cov_dim)
        self.plugin_estmr = PricingPlugIn(cov_dim=cov_dim)  # plugin estimator

    def fit(self, covariates: np.ndarray, prices: np.ndarray, outcomes: np.ndarray, n_bootstraps: int = 100):
        # fill in placeholders
        self.n_bootstraps = n_bootstraps

        # assign each customer to a group
        group_arr = np.array([self.group_func(x) for x in X])  # (n_obs, )

        # bootstrap sample indices, shape (num_bootstraps, m)
        boot_index_arr = np.random.choice(np.arange(X.shape[0]), size=(self.n_bootstraps, X.shape[0]), replace=True)  
        
        boot_y_arr = Y.flatten()[boot_index_arr]
        boot_t_arr = T.flatten()[boot_index_arr]
        boot_group_arr = group_arr.flatten()[boot_index_arr]

        self.boot_te_arr = np.array([te_with_dim(
            group=boot_group_arr[boot_id, :], t=boot_t_arr[boot_id, :], y=boot_y_arr[boot_id, :], n_groups=2
        ) for boot_id in range(self.n_bootstraps)])

        self.plugin_estimator.fit(X, T, Y)

        return self
    
    def estimate_targeting_value(
        self, X: np.ndarray, budget: int = 1, resid_method: float = 2
    ) -> np.ndarray:
        """   
        Given a set of covariates, estimate the treatment effect for each covariate.

        Params:
        -------
        X: np.ndarray, shape (M, 1), the covariates
        budget: int, the number of customers to target
        resid_method: int, the method to calculate the empirical estimation error

        Returns:
        -------
        np.ndarray, shape (n_bootstraps, n_oobs)
        """
        # assign each customer to a group
        group_arr = np.array([self.group_func(x) for x in X])  # (n_obs, )

        # create an array to store the treatment effect estimates for each bootstrap sample
        boot_est_arr = self.boot_te_arr[:, group_arr]  # (n_bootstraps, n_obs)

        # optimize targeting for each bootstrap sample
        boot_targ_val_arr = -np.sort(-boot_est_arr, axis=1)[:, :budget].sum(axis=1)  # (n_bootstraps, )

        # calculate plugin estimate
        plugin_estimate = self.plugin_estimator.estimate_targeting_value(X, budget=budget)

        # calculate the corrected treatment effect estimate
        return 2 * plugin_estimate - boot_targ_val_arr.mean()